# Notebook 03 — Побудова ізохрон пішохідної доступності

**Ізохрона** — геометрична фігура, що охоплює всі точки, досяжні від заданої точки за певний час.

Методологія:
1. Завантажуємо пішохідну вуличну мережу Києва
2. Для кожної лікарні/школи будуємо ізохрони 10 хв і 30 хв пішки
3. Знаходимо всі зупинки GTFS у межах кожної ізохрони
4. Зберігаємо результати для підрахунку індексу доступності

**Швидкість пішохода:** 4.5 км/год → 10 хв ≈ 750 м, 30 хв ≈ 2250 м


In [ ]:
from config_loader import cfg, CITY, WALK_10MIN_M, WALK_30MIN_M, CRS_METRIC
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point, MultiPoint, Polygon, LineString
from shapely.ops import unary_union
from pyproj import Transformer
import numpy as np
import os
import pickle
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

print('Бібліотеки завантажено')
print(f'osmnx: {ox.__version__}, networkx: {nx.__version__}')

In [ ]:
GTFS_PATH = cfg['paths']['gtfs']
gpkg_data_path = '../data/osm/osm_data.gpkg'

# OSM дані
osm_gstops = gpd.read_file(gpkg_data_path, layer='stops')
osm_ghospitals = gpd.read_file(gpkg_data_path, layer='hospitals')
osm_gschools = gpd.read_file(gpkg_data_path, layer='schools')
osm_gmetro = gpd.read_file(gpkg_data_path, layer='metro')
osm_gtram = gpd.read_file(gpkg_data_path, layer='tram')

osm: dict[str, gpd.GeoDataFrame] = {
    'gstops': osm_gstops,
    'ghospitals': osm_ghospitals,
    'gschools': osm_gschools,
    'gmetro': osm_gmetro,
    'gtram': osm_gtram
}

# Додаємо тип об'єкта та єдиний ідентифікатор
osm['ghospitals']['facility_type'] = 'hospital'
osm['ghospitals']['facility_id'] = ['H' + str(i) for i in range(len(osm['ghospitals']))]

osm['gschools']['facility_type'] = 'school'
osm['gschools']['facility_id'] = ['S' + str(i) for i in range(len(osm['gschools']))]

# Об'єднуємо в один датафрейм
ghospital_schools = pd.concat([osm['ghospitals'], osm['gschools']], ignore_index=True)
ghospital_schools = gpd.GeoDataFrame(ghospital_schools, crs='EPSG:4326')

display(ghospital_schools)

# Додаємо унікальний ідентифікатор для metro (для підрахунку n_stops)
metro_gdf = osm['gmetro'][['geometry']].copy()
metro_gdf['metro_stop_id'] = ['METRO_' + str(i) for i in range(len(metro_gdf))]

print(f'OSM зупинки (фізичні):            {len(osm["gstops"])}')
print(f'Станції метро (OSM):              {len(metro_gdf)}')
print(f'Лікарні та клініки:               {len(osm["ghospitals"])}')
print(f'Школи:                            {len(osm["gschools"])}')


## 1. Завантаження пішохідної вуличної мережі

Використовуємо osmnx для завантаження графу пішохідних шляхів Києва.
Граф зберігаємо локально, щоб не завантажувати повторно.

In [ ]:
GRAPH_PATH = '../data/osm/kyiv_walk_graph.pkl'

if os.path.exists(GRAPH_PATH):
    print('Граф знайдено локально, завантажуємо...')
    with open(GRAPH_PATH, 'rb') as f:
        G = pickle.load(f)
    print('Граф завантажено з кешу')
else:
    print('Завантаження пішохідної мережі Києва з OSM...')
    G = ox.graph_from_place('Kyiv, Ukraine', network_type='walk', simplify=True)
    # Зберігаємо для повторного використання
    with open(GRAPH_PATH, 'wb') as f:
        pickle.dump(G, f)
    print('Граф збережено')

print(f'Вузлів: {G.number_of_nodes():,}')
print(f'Ребер: {G.number_of_edges():,}')
# fig, ax = ox.plot_graph(G)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from shapely.geometry import LineString

# ── Дві точки на Подолі ──────────────────────────────────────────────────────
orig = ox.distance.nearest_nodes(G, X=30.5155, Y=50.4615)   # старт
dest = ox.distance.nearest_nodes(G, X=30.5225, Y=50.4670)   # фінал

# ── Алгоритм Дейкстри ────────────────────────────────────────────────────────
# Відстань від старту до фіналу (і всіх проміжних вузлів)
dist_to_dest, _ = nx.single_source_dijkstra(G, orig, dest, weight='length')

# Всі вузли, які Дейкстра відвідав до того як дістався до фіналу
all_distances = nx.single_source_dijkstra_path_length(G, orig, cutoff=dist_to_dest, weight='length')
visited_nodes = set(all_distances.keys())

# Фінальний найкоротший шлях
route = nx.shortest_path(G, orig, dest, weight='length')
route_edges = list(zip(route[:-1], route[1:]))

# ── Підграф для фону (трохи більший за зону пошуку) ──────────────────────────
mid_x = (G.nodes[orig]['x'] + G.nodes[dest]['x']) / 2
mid_y = (G.nodes[orig]['y'] + G.nodes[dest]['y']) / 2
center_bg = ox.distance.nearest_nodes(G, X=mid_x, Y=mid_y)
bg_radius = dist_to_dest * 1.15
bg_subgraph = nx.ego_graph(G, center_bg, radius=bg_radius, distance='length')

# ── Малюємо ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 11), facecolor='#fff')
ax.set_facecolor('#fff')

# 1. Фонові ребра (вся мережа навколо)
for u, v, data in bg_subgraph.edges(data=True):
    geom = data.get('geometry')
    if geom is None:
        xs = [G.nodes[u]['x'], G.nodes[v]['x']]
        ys = [G.nodes[u]['y'], G.nodes[v]['y']]
    else:
        xs, ys = geom.xy
    ax.plot(xs, ys, color='#2a3a4a', linewidth=0.8, zorder=1)

# 2. Відвідані вузли (помаранчевий ореол — "фронт" Дейкстри)
vis_x = [G.nodes[n]['x'] for n in visited_nodes if n not in route]
vis_y = [G.nodes[n]['y'] for n in visited_nodes if n not in route]
ax.scatter(vis_x, vis_y, s=18, c='#000', alpha=0.55, zorder=2, linewidths=0)

# 3. Ребра найкоротшого шляху
for u, v in route_edges:
    data = G.get_edge_data(u, v)
    # MultiDiGraph може мати кілька ребер між вузлами — беремо перше
    edge_data = list(data.values())[0] if data else {}
    geom = edge_data.get('geometry')
    if geom is None:
        xs = [G.nodes[u]['x'], G.nodes[v]['x']]
        ys = [G.nodes[u]['y'], G.nodes[v]['y']]
    else:
        xs, ys = geom.xy
    ax.plot(xs, ys, color='#ef5350', linewidth=3.5, zorder=4, solid_capstyle='round')

# 4. Вузли маршруту
route_x = [G.nodes[n]['x'] for n in route[1:-1]]
route_y = [G.nodes[n]['y'] for n in route[1:-1]]
ax.scatter(route_x, route_y, s=35, c='#ef9a9a', zorder=5, linewidths=0)

# 5. Старт і фінал
ax.scatter([G.nodes[orig]['x']], [G.nodes[orig]['y']],
           s=220, c='#66bb6a', zorder=6, linewidths=0)
ax.scatter([G.nodes[dest]['x']], [G.nodes[dest]['y']],
           s=220, c='#ffee58', marker='*', zorder=6, linewidths=0)

# ── Легенда ──────────────────────────────────────────────────────────────────
bg_patch      = mlines.Line2D([], [], color='#2a3a4a', linewidth=1.5, label='Вулична мережа')
visited_patch = mpatches.Patch(color='#ff9800', alpha=0.7, label=f'Відвідані вузли ({len(visited_nodes):,} шт.)')
route_line    = mlines.Line2D([], [], color='#ef5350', linewidth=3, label=f'Найкоротший шлях ({dist_to_dest:.0f} м)')
orig_patch    = mpatches.Patch(color='#66bb6a', label='Старт')
dest_patch    = mpatches.Patch(color='#ffee58', label='Фінал')

ax.legend(handles=[bg_patch, visited_patch, route_line, orig_patch, dest_patch],
          loc='lower right', framealpha=0.4, labelcolor='white',
          facecolor='#0d1117', edgecolor='#334466', fontsize=10)

ax.set_title(
    'Алгоритм Дейкстри — пошук найкоротшого шляху\nПішохідна мережа Києва (район Поділ)',
    color='white', fontsize=13, pad=14
)
ax.tick_params(colors='#555577')
for spine in ax.spines.values():
    spine.set_edgecolor('#222244')

plt.tight_layout()
out_path = '../outputs/dijkstra_algorithm.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Збережено: {out_path}')
print(f'Довжина маршруту:   {dist_to_dest:.0f} м')
print(f'Вузлів у маршруті:  {len(route)}')
print(f'Відвідано Дейкстрою: {len(visited_nodes):,} вузлів')

In [ ]:
# Проектуємо граф у метричну CRS (UTM зона 36N для Київа)
print('Проектуємо граф у EPSG:32636...')
display(G)
G_proj = ox.project_graph(G, to_crs=CRS_METRIC)
print(f'CRS графу: {G_proj.graph["crs"]}')

display(G)
# Зберігаємо WGS84 координати до проекції (для збереження у р езультатах)
ghospital_schools['lon_wgs84'] = ghospital_schools.geometry.x
ghospital_schools['lat_wgs84'] = ghospital_schools.geometry.y
display(G)

# Проектуємо об'єкти соцінфраструктури та зупинки
facilities_proj = ghospital_schools.to_crs('EPSG:32636')
stops_proj = osm['gstops'].to_crs('EPSG:32636')
osm_stops_proj = osm['gstops'].to_crs('EPSG:32636')  # OSM  — для підрахунку n_stops
metro_proj = metro_gdf.to_crs('EPSG:32636')          # Метро — додається до n_stops

print('Проекція виконана')

## 2. Функція побудови ізохрони

Алгоритм:
1. Знаходимо найближчий вузол графу до об'єкта
2. Запускаємо алгоритм Дейкстри для знаходження всіх вузлів у радіусі `radius` метрів
3. Будуємо підграф з досяжних вузлів та збираємо всі ребра
4. Буферизуємо кожне ребро на `EDGE_BUFFER_M` метрів → `unary_union` → справжня ізохрона

.

In [ ]:
# Буфер навколо кожного ребра графу — "ширина коридору" вулиці
# 25 м достатньо щоб полігони сусідніх вулиць зімкнулись у суцільну зону
EDGE_BUFFER_M = 25

def build_isochrones_both(G_proj, lon, lat, radius_small, radius_large):
    """
    Будує справжні ізохрони для ДВОХ радіусів за ОДНИМ запуском алгоритму Дейкстри.

    Метод: буферизація досяжних ребер мережі + unary_union.
    На відміну від Convex Hull, полігон точно відповідає формі вуличної мережі
    і не включає недосяжні ділянки (парки, річки, внутрішні двори).

    Повертає: (iso_small, iso_large) — два Shapely Polygon/MultiPolygon
    """
    # Крок 1: знаходимо найближчий вузол (один раз)
    center_node = ox.distance.nearest_nodes(G_proj, X=lon, Y=lat)

    # Крок 2: Дейкстра — один запуск покриває обидва радіуси
    distances = nx.single_source_dijkstra_path_length(
        G_proj, center_node, cutoff=radius_large, weight='length'
    )

    # Крок 3: розділяємо вузли по порогах
    nodes_small = {n for n, d in distances.items() if d <= radius_small}
    nodes_large = set(distances.keys())

    def nodes_to_isochrone(nodes_set, fallback_r):
        if not nodes_set:
            return Point(lon, lat).buffer(fallback_r)

        # Підграф лише з досяжних вузлів — networkx повертає вигляд (без копії)
        subgraph = G_proj.subgraph(nodes_set)

        # Збираємо геометрії всіх ребер підграфу
        edge_geoms = []
        for u, v, data in subgraph.edges(data=True):
            geom = data.get('geometry')
            if geom is None:
                # Ребро без збереженої геометрії → пряма лінія між вузлами
                geom = LineString([
                    (G_proj.nodes[u]['x'], G_proj.nodes[u]['y']),
                    (G_proj.nodes[v]['x'], G_proj.nodes[v]['y'])
                ])
            edge_geoms.append(geom)

        if not edge_geoms:
            # Ізольований вузол (немає ребер) → кругова зона
            pts = [Point(G_proj.nodes[n]['x'], G_proj.nodes[n]['y']) for n in nodes_set]
            return unary_union([p.buffer(EDGE_BUFFER_M) for p in pts])

        # Буферизуємо кожне ребро та об'єднуємо в один полігон
        return unary_union([g.buffer(EDGE_BUFFER_M) for g in edge_geoms])

    return nodes_to_isochrone(nodes_small, radius_small), nodes_to_isochrone(nodes_large, radius_large)


print('Функції ізохрон визначено')
print(f'Метод: буферизація ребер графу ({EDGE_BUFFER_M} м) + unary_union')
print('Результат: справжня ізохрона по вуличній мережі (не Convex Hull)')

## 3. Обчислення ізохрон для всіх об'єктів

**Параметри:**
- 10 хвилин пішки = 750 метрів
- 30 хвилин пішки = 2250 метрів

Результати зберігаємо поступово — щоб не втратити прогрес при можливих помилках мережі.

In [ ]:
results = []
errors = []
success_count = 0
error_count = 0

print(f"Будуємо ізохрони для {len(facilities_proj)} об'єктів...")

progress = tqdm(
    facilities_proj.iterrows(),
    total=len(facilities_proj),
    desc="Ізохрони"
)

for idx, row in progress:
    try:
        lon_proj = row.geometry.x
        lat_proj = row.geometry.y

        iso_10, iso_30 = build_isochrones_both(
            G_proj, lon_proj, lat_proj, WALK_10MIN_M, WALK_30MIN_M
        )

        # OSM зупинки + метро в ізохронах
        osm_in_30 = osm_stops_proj[osm_stops_proj.within(iso_30)]
        osm_in_10 = osm_in_30[osm_in_30.within(iso_10)]
        metro_in_10 = metro_proj[metro_proj.within(iso_10)]
        metro_in_30 = metro_proj[metro_proj.within(iso_30)]

        results.append({
            'facility_id':   row['facility_id'],
            'facility_type': row['facility_type'],
            'name':          row.get('name', ''),
            'lon':           row['lon_wgs84'],
            'lat':           row['lat_wgs84'],
            'n_stops_10min': len(osm_in_10) + len(metro_in_10),
            'n_stops_30min': len(osm_in_30) + len(metro_in_30),
        })
        success_count += 1

    except Exception as e:
        errors.append({'facility_id': row.get('facility_id', str(idx)), 'error': str(e)})
        error_count += 1

    progress.set_postfix({'success': success_count, 'errors': error_count})

print(f"\nГотово! Успішно: {success_count}, з помилкою: {error_count}")
if errors:
    print("Перші помилки:")
    for e in errors[:3]:
        print(f"  {e['facility_id']} → {e['error']}")


In [ ]:
results_df = pd.DataFrame(results)
results_df.to_parquet('../data/processed/isochrone_results.parquet', index=False)

print('Результати збережено: data/processed/isochrone_results.parquet')
print(f'Рядків: {len(results_df)}')
print()
print('=== Статистика ===')
print(f'Середня кількість зупинок OSM (10 хв): {results_df["n_stops_10min"].mean():.1f}')
print(f'Середня кількість зупинок OSM (30 хв): {results_df["n_stops_30min"].mean():.1f}')
print(f"Об'єктів без жодної зупинки (10 хв): {(results_df['n_stops_10min'] == 0).sum()}")
print(f"Об'єктів без жодної зупинки (30 хв): {(results_df['n_stops_30min'] == 0).sum()}")


## 4. Візуалізація прикладу ізохрони

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
import pickle

# ─── Fallback: results_df ────────────────────────────────────────────────────
if 'results_df' not in vars():
    results_df = pd.read_parquet('../data/processed/isochrone_results.parquet')
    print(f'results_df завантажено з файлу: {len(results_df)} рядків')

# ─── Fallback: G_proj ────────────────────────────────────────────────────────
if 'G_proj' not in vars():
    GRAPH_PATH = '../data/osm/kyiv_walk_graph.pkl'
    with open(GRAPH_PATH, 'rb') as _f:
        _G_raw = pickle.load(_f)
    G_proj = ox.projection.project_graph(_G_raw, to_crs='EPSG:32636')
    print('G_proj завантажено та спроектовано')

# ─── Fallback: facilities_proj ───────────────────────────────────────────────
if 'facilities_proj' not in vars():
    _gpkg = '../data/osm/osm_data.gpkg'
    _h = gpd.read_file(_gpkg, layer='hospitals'); _h['facility_type'] = 'hospital'
    _s = gpd.read_file(_gpkg, layer='schools');   _s['facility_type'] = 'school'
    facilities_proj = gpd.GeoDataFrame(
        pd.concat([_h, _s], ignore_index=True), crs='EPSG:4326'
    ).to_crs('EPSG:32636')
    print(f'facilities_proj: {len(facilities_proj)} об\'єктів')

# ─── Fallback: osm_gstops ────────────────────────────────────────────────────
if 'osm_gstops' not in vars():
    osm_gstops = gpd.read_file('../data/osm/osm_data.gpkg', layer='stops')

# ─── Fallback: build_isochrones_both (якщо клітинка 03-isochrone-function не виконана) ──
if 'build_isochrones_both' not in vars():
    def build_isochrones_both(G, x, y, r10, r30, buffer_m=25):
        from shapely.ops import unary_union as _uu
        from shapely.geometry import LineString as _LS
        center = ox.distance.nearest_nodes(G, X=x, Y=y)
        dists  = nx.single_source_dijkstra_path_length(G, center, cutoff=r30, weight='length')
        def _make(nodes_set, fallback_r):
            if not nodes_set:
                return Point(x, y).buffer(fallback_r)
            sub   = G.subgraph(nodes_set)
            geoms = []
            for u, v, data in sub.edges(data=True):
                g = data.get('geometry') or _LS([(G.nodes[u]['x'], G.nodes[u]['y']),
                                                  (G.nodes[v]['x'], G.nodes[v]['y'])])
                geoms.append(g)
            if not geoms:
                return _uu([Point(G.nodes[n]['x'], G.nodes[n]['y']).buffer(buffer_m) for n in nodes_set])
            return _uu([g.buffer(buffer_m) for g in geoms])
        return (_make({n for n, d in dists.items() if d <= r10}, r10),
                _make(set(dists.keys()), r30))
    print('build_isochrones_both визначено локально')

# ─── Вибір об'єкта для прикладу ──────────────────────────────────────────────
ex       = facilities_proj.iloc[0]
lon_p, lat_p = ex.geometry.x, ex.geometry.y
name_str = str(ex.get('name', 'Об\'єкт'))
ftype_ua = 'Лікарня/клініка' if ex.get('facility_type') == 'hospital' else 'Школа'
lat_wgs  = float(results_df.iloc[0]['lat'])
lon_wgs  = float(results_df.iloc[0]['lon'])

iso_10_proj, iso_30_proj = build_isochrones_both(G_proj, lon_p, lat_p, WALK_10MIN_M, WALK_30MIN_M)

# Конвертація в WGS84 для folium
iso_10_wgs = gpd.GeoSeries([iso_10_proj], crs='EPSG:32636').to_crs('EPSG:4326').iloc[0]
iso_30_wgs = gpd.GeoSeries([iso_30_proj], crs='EPSG:32636').to_crs('EPSG:4326').iloc[0]

# Зупинки всередині ізохрон (WGS84)
stops_in_10   = osm_gstops[osm_gstops.within(iso_10_wgs)].copy()
stops_in_30   = osm_gstops[osm_gstops.within(iso_30_wgs)].copy()
stops_ring_30 = stops_in_30[~stops_in_30.index.isin(stops_in_10.index)]

# ─── 1. Статичний PNG (matplotlib) ───────────────────────────────────────────
_sp = osm_gstops.to_crs('EPSG:32636')
fig, ax = plt.subplots(figsize=(10, 8))
if not iso_30_proj.is_empty:
    gpd.GeoSeries([iso_30_proj], crs='EPSG:32636').plot(
        ax=ax, color='#FFF3E0', edgecolor='#FF9800', linewidth=2, alpha=0.6)
if not iso_10_proj.is_empty:
    gpd.GeoSeries([iso_10_proj], crs='EPSG:32636').plot(
        ax=ax, color='#E3F2FD', edgecolor='#2196F3', linewidth=2, alpha=0.6)
_sp[_sp.index.isin(stops_in_10.index)].plot(ax=ax, color='#2196F3', markersize=20, zorder=4)
_sp[_sp.index.isin(stops_ring_30.index)].plot(ax=ax, color='#FF9800', markersize=12, zorder=4)
ax.plot(lon_p, lat_p, 'r*', markersize=18, zorder=5)
ax.legend(handles=[
    mpatches.Patch(facecolor='#FFF3E0', edgecolor='#FF9800', label=f'Ізохрона 30 хв ({WALK_30MIN_M} м)'),
    mpatches.Patch(facecolor='#E3F2FD', edgecolor='#2196F3', label=f'Ізохрона 10 хв ({WALK_10MIN_M} м)'),
], loc='upper right')
ax.set_title(f'Ізохрони пішохідної доступності\n{name_str[:60]}', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/05_isochrone_example.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── 2. Інтерактивна карта міста (folium) ────────────────────────────────────
m = folium.Map(location=[lat_wgs, lon_wgs], zoom_start=15, tiles='CartoDB positron')

if not iso_30_wgs.is_empty:
    folium.GeoJson(
        iso_30_wgs.__geo_interface__,
        name='Ізохрона 30 хв',
        style_function=lambda _: {
            'fillColor': '#FF9800', 'color': '#E65100',
            'weight': 2.5, 'fillOpacity': 0.18,
        },
        tooltip='Пішки 30 хв'
    ).add_to(m)

if not iso_10_wgs.is_empty:
    folium.GeoJson(
        iso_10_wgs.__geo_interface__,
        name='Ізохрона 10 хв',
        style_function=lambda _: {
            'fillColor': '#2196F3', 'color': '#1565C0',
            'weight': 2.5, 'fillOpacity': 0.28,
        },
        tooltip='Пішки 10 хв'
    ).add_to(m)

# Зупинки у зоні 10 хв — сині кола
for _, stop in stops_in_10.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=6, color='#1565C0', weight=1.5,
        fill=True, fill_color='#2196F3', fill_opacity=0.9,
        tooltip=str(stop.get('name', '')) or 'Зупинка',
    ).add_to(m)

# Зупинки у зоні 10–30 хв — помаранчеві кола
for _, stop in stops_ring_30.iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=5, color='#E65100', weight=1.5,
        fill=True, fill_color='#FF9800', fill_opacity=0.75,
        tooltip=str(stop.get('name', '')) or 'Зупинка',
    ).add_to(m)

# Маркер об'єкта
folium.Marker(
    location=[lat_wgs, lon_wgs],
    popup=folium.Popup(
        f'<b>{name_str}</b><br>'
        f'{ftype_ua}<br>'
        f'Зупинок (10 хв): <b>{len(stops_in_10)}</b><br>'
        f'Зупинок (30 хв): <b>{len(stops_in_30)}</b>',
        max_width=280
    ),
    tooltip=name_str[:45],
    icon=folium.Icon(
        color='red',
        icon='plus-sign' if ex.get('facility_type') == 'hospital' else 'book',
        prefix='glyphicon'
    )
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save('../outputs/05_isochrone_example.html')

print(f'Карта збережена:  outputs/05_isochrone_example.html')
print(f'PNG збережено:    outputs/05_isochrone_example.png')
print(f'Об\'єкт:          {name_str[:60]}')
print(f'Зупинок 10 хв:   {len(stops_in_10)}')
print(f'Зупинок 30 хв:   {len(stops_in_30)}')
print()
print('Notebook 03 виконано. Переходьте до 04_accessibility_index.ipynb')
m